# Phase 4 Full Prescription Cropper and Annotation Tools

This notebook prepares custom word crops for TrOCR training and thesis evaluation.

## Recommended use in the final pipeline

1. Use `phase4_region_line_segmentation_colab.ipynb` to create `data/final_region_hybrid_line_word/word_manifest.csv` from raw full-prescription images.
2. Use this notebook when you need to review, label, or correct those word crops.
3. Export reviewed word crops into a TrOCR-ready dataset.
4. Create a train-only augmented copy of that word dataset for OCR fine-tuning.

Region detection is already handled by the strong YOLO handwritten-region model. Line and word crops are produced by the hybrid segmentation scripts; the annotation UI here is for improving labels and crop quality before OCR training.


## Step 0: Mount Drive and Set Project Root

Run the next cell first in Colab. It mounts Google Drive, clones or pulls the GitHub repository into `/content/drive/MyDrive/phase4_project/repo`, and uses `repo/data` for raw images and generated files. Put raw prescription images in `data/raw` inside that Drive-backed repo folder.


In [ ]:
# Colab/Drive project setup.
# This cell mounts Google Drive, keeps the repository in Drive, and keeps data in Drive.
# Put raw prescriptions here after this cell runs:
# /content/drive/MyDrive/phase4_project/repo/data/raw

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/nbl-ahmd/project.git'
DRIVE_BASE = Path('/content/drive/MyDrive/phase4_project')
REPO_DIR = DRIVE_BASE / 'repo'

IN_COLAB = 'google.colab' in sys.modules or 'COLAB_GPU' in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE.mkdir(parents=True, exist_ok=True)

    if not (REPO_DIR / 'pipeline').exists():
        if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
            raise RuntimeError(
                f'{REPO_DIR} exists but does not look like the project repo. '
                'Move/rename it or set REPO_DIR to the correct folder.'
            )
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)
else:
    # Local Jupyter/VS Code fallback: use the current repository folder.
    REPO_DIR = Path.cwd()

os.chdir(REPO_DIR)

# data/ is gitignored, so it is safe to keep raw images and generated files here.
# In Colab this folder is inside Google Drive and persists across sessions.
DATA_DIR = REPO_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents=True, exist_ok=True)

print('IN_COLAB:', IN_COLAB)
print('Repository:', REPO_DIR)
print('Data folder:', DATA_DIR)
print('Current working directory:', Path.cwd())
print('Has pipeline:', (Path.cwd() / 'pipeline').exists())
print('Raw data folder:', RAW_DIR)
print('Raw images currently:', len([p for p in RAW_DIR.glob('*') if p.is_file()]))

## Step 1: Install Dependencies

This installs the image preprocessing, segmentation, dataframe, and notebook widget dependencies. TrOCR dependencies are not required for annotation; install them later in the training/final inference notebook.


In [ ]:
# Core dependencies for preprocessing, cropping, segmentation, and annotation CSV handling.
!python3 -m pip install -q -r pipeline/requirements.txt ipywidgets


## Step 2: Configure Paths

All generated files go under `data/processed`. You can change these paths, but keeping them stable makes the later TrOCR notebook easier to run.


In [ ]:
# Central path configuration used by all later cells.
RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
PAGES_DIR = PROCESSED_DIR / 'pages'
PAGE_MANIFEST = PROCESSED_DIR / 'page_manifest.csv'
LAYOUT_PACKAGE_DIR = PROCESSED_DIR / 'layout_annotation_package'
YOLO_LABELS_DIR = PROCESSED_DIR / 'layout_yolo_labels'
REGIONS_DIR = PROCESSED_DIR / 'regions'
REGION_MANIFEST = PROCESSED_DIR / 'region_manifest.csv'
LINE_CROPS_DIR = PROCESSED_DIR / 'line_crops'
LINE_MANIFEST = PROCESSED_DIR / 'line_manifest.csv'
WORD_CROPS_DIR = PROCESSED_DIR / 'word_crops'
WORD_MANIFEST = PROCESSED_DIR / 'word_manifest.csv'
WORD_ANNOTATIONS = PROCESSED_DIR / 'word_annotations.csv'
CUSTOM_WORD_DATASET = Path('data/custom_word_ocr_dataset')
AUGMENTED_WORD_DATASET = Path('data/custom_word_ocr_dataset_augmented')

for folder in [PROCESSED_DIR, PAGES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Raw images:', len(list(RAW_DIR.glob('*'))))
print('Processed output:', PROCESSED_DIR)


## Step 3: Preprocess Full Prescription Images

This converts raw photos into cleaner page images using resizing, deskewing, denoising, and contrast enhancement.


In [ ]:
# Preprocess all full prescription images from data/raw.
# Output: cleaned page images + page_manifest.csv.
!python3 pipeline/scripts/preprocess_pages.py \
  --input-dir "{RAW_DIR}" \
  --output-dir "{PAGES_DIR}" \
  --manifest-out "{PAGE_MANIFEST}"


## Step 4: Preview Preprocessed Pages

Use this preview to confirm the raw prescriptions loaded correctly before annotation or segmentation.


In [ ]:
# Display a few processed pages so you can visually check image quality.
from IPython.display import display, Image as IPImage

sample_pages = sorted(PAGES_DIR.glob('*'))[:5]
print('Previewing', len(sample_pages), 'processed pages')
for page in sample_pages:
    print(page)
    display(IPImage(filename=str(page), width=450))


## Step 5: Prepare Layout Annotation Package

This creates a folder that can be uploaded to CVAT or Label Studio. Annotate these classes in this order:

1. `header`
2. `handwritten_region`
3. `footer`

After annotation, export YOLO labels and place the `.txt` files in `data/processed/layout_yolo_labels`.


In [ ]:
# Prepare images and class file for CVAT/Label Studio layout annotation.
!python3 pipeline/scripts/prepare_layout_annotation.py \
  --pages-dir "{PAGES_DIR}" \
  --output-dir "{LAYOUT_PACKAGE_DIR}" \
  --copy-images

print('Upload this folder to CVAT/Label Studio:', LAYOUT_PACKAGE_DIR)
print('After annotation, put YOLO .txt labels here:', YOLO_LABELS_DIR)


## Step 6A: Crop Handwritten Regions from YOLO Labels

Run this after exporting YOLO labels from CVAT/Label Studio. This is the recommended final-evaluation route because it follows the paper's region segmentation stage.


In [ ]:
# Crop handwritten regions using manually annotated YOLO labels.
# Class order comes from pipeline/config/layout_classes.txt.
if YOLO_LABELS_DIR.exists() and any(YOLO_LABELS_DIR.glob('*.txt')):
    !python3 pipeline/scripts/crop_regions_from_yolo.py \
      --pages-dir "{PAGES_DIR}" \
      --labels-dir "{YOLO_LABELS_DIR}" \
      --class-map pipeline/config/layout_classes.txt \
      --target-label handwritten_region \
      --output-dir "{REGIONS_DIR}" \
      --manifest-out "{REGION_MANIFEST}"
else:
    print('YOLO labels not found yet:', YOLO_LABELS_DIR)
    print('Use Step 6B for quick heuristic crop generation, or export labels from CVAT first.')


## Step 6B: Quick Heuristic Word Crop Generation Without YOLO Labels

Use this only for fast demos or early annotation. It estimates the handwritten region automatically, then segments lines and words. For final evaluation, YOLO/CVAT labels are cleaner.


In [ ]:
# Optional quick route: generates page, region, line, and word manifests for annotation.
# If models/region_yolo_best.pt exists, this uses it for region detection; otherwise it falls back to a rough proposal.
RUN_QUICK_SEGMENTATION_ROUTE = False
REGION_MODEL = Path('models/region_yolo_best.pt')

if RUN_QUICK_SEGMENTATION_ROUTE:
    extra_region_args = f'--yolo-model {REGION_MODEL} --target-class 0' if REGION_MODEL.exists() else ''
    !python3 pipeline/scripts/run_end_to_end.py \
      --input "{RAW_DIR}" \
      --output-dir data/processed/hybrid_word_pipeline \
      {extra_region_args} \
      --ocr-backend none \
      --ocr-unit word \
      --line-padding 6
    print('Hybrid word manifest:', Path('data/processed/hybrid_word_pipeline/word_manifest.csv'))
else:
    print('Set RUN_QUICK_SEGMENTATION_ROUTE = True if you need automatic word crops for annotation.')


## Step 6C: Manual Handwritten-Region Cropper Inside Notebook

Use this if you want to avoid CVAT/Label Studio. It shows one processed page at a time and lets you choose the handwritten region bounding box using sliders. Click **Save Region** to crop the region and append it to `region_manifest.csv`.

This is simpler than CVAT, but enough for final-project demos when the main goal is to generate clean handwritten regions for line and word segmentation.


In [ ]:
# Manual in-notebook region cropper.
# Use the sliders to select the handwritten area, then save the crop to data/processed/regions.
import csv
import cv2
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage

manual_pages = sorted(PAGES_DIR.glob('*'))
if not manual_pages:
    raise FileNotFoundError(f'No processed pages found in {PAGES_DIR}. Run Step 3 first.')

REGIONS_DIR.mkdir(parents=True, exist_ok=True)
REGION_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

manual_state = {'page_idx': 0, 'region_idx': 0}
manual_out = widgets.Output()
page_label = widgets.Label()
preview_out = widgets.Output()

x1_slider = widgets.IntSlider(description='x1', min=0, max=100, value=0, continuous_update=False)
y1_slider = widgets.IntSlider(description='y1', min=0, max=100, value=0, continuous_update=False)
x2_slider = widgets.IntSlider(description='x2', min=1, max=100, value=100, continuous_update=False)
y2_slider = widgets.IntSlider(description='y2', min=1, max=100, value=100, continuous_update=False)
prev_page_btn = widgets.Button(description='Previous Page')
next_page_btn = widgets.Button(description='Next Page')
preview_btn = widgets.Button(description='Preview Crop')
save_region_btn = widgets.Button(description='Save Region', button_style='success')


def current_page_path():
    return manual_pages[manual_state['page_idx']]


def load_page_sliders():
    page_path = current_page_path()
    image = cv2.imread(str(page_path))
    if image is None:
        raise ValueError(f'Could not read image: {page_path}')
    h, w = image.shape[:2]
    x1_slider.max = w - 1
    x2_slider.max = w
    y1_slider.max = h - 1
    y2_slider.max = h
    x1_slider.value = 0
    y1_slider.value = int(h * 0.15)
    x2_slider.value = w
    y2_slider.value = int(h * 0.9)
    page_label.value = f'Page {manual_state["page_idx"] + 1}/{len(manual_pages)}: {page_path.name} ({w}x{h})'
    show_preview(full_page=True)


def show_preview(full_page=False):
    page_path = current_page_path()
    image = cv2.imread(str(page_path))
    h, w = image.shape[:2]
    x1 = max(0, min(x1_slider.value, w - 1))
    y1 = max(0, min(y1_slider.value, h - 1))
    x2 = max(x1 + 1, min(x2_slider.value, w))
    y2 = max(y1 + 1, min(y2_slider.value, h))
    vis = image.copy()
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 220, 0), 4)
    tmp_path = PROCESSED_DIR / '_manual_region_preview.jpg'
    if full_page:
        cv2.imwrite(str(tmp_path), vis)
    else:
        crop = image[y1:y2, x1:x2]
        cv2.imwrite(str(tmp_path), crop)
    with preview_out:
        clear_output(wait=True)
        display(IPImage(filename=str(tmp_path), width=650))


def append_region_manifest(row):
    columns = [
        'region_id', 'page_id', 'region_label', 'page_image_path', 'region_image_path',
        'x1_page', 'y1_page', 'x2_page', 'y2_page', 'width', 'height'
    ]
    exists = REGION_MANIFEST.exists()
    with REGION_MANIFEST.open('a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        if not exists:
            writer.writeheader()
        writer.writerow(row)


def save_region(_):
    page_path = current_page_path()
    image = cv2.imread(str(page_path))
    h, w = image.shape[:2]
    x1 = max(0, min(x1_slider.value, w - 1))
    y1 = max(0, min(y1_slider.value, h - 1))
    x2 = max(x1 + 1, min(x2_slider.value, w))
    y2 = max(y1 + 1, min(y2_slider.value, h))
    page_id = page_path.stem
    region_id = f'{page_id}_manual_r{manual_state["region_idx"]:02d}'
    region_path = REGIONS_DIR / f'{region_id}.png'
    cv2.imwrite(str(region_path), image[y1:y2, x1:x2])
    append_region_manifest({
        'region_id': region_id,
        'page_id': page_id,
        'region_label': 'handwritten_region',
        'page_image_path': str(page_path),
        'region_image_path': str(region_path),
        'x1_page': x1,
        'y1_page': y1,
        'x2_page': x2,
        'y2_page': y2,
        'width': x2 - x1,
        'height': y2 - y1,
    })
    manual_state['region_idx'] += 1
    with manual_out:
        clear_output(wait=True)
        print('Saved:', region_path)
        print('Updated:', REGION_MANIFEST)


def prev_page(_):
    manual_state['page_idx'] = max(0, manual_state['page_idx'] - 1)
    load_page_sliders()


def next_page(_):
    manual_state['page_idx'] = min(len(manual_pages) - 1, manual_state['page_idx'] + 1)
    load_page_sliders()

prev_page_btn.on_click(prev_page)
next_page_btn.on_click(next_page)
preview_btn.on_click(lambda _: show_preview(full_page=False))
save_region_btn.on_click(save_region)

manual_ui = widgets.VBox([
    page_label,
    widgets.HBox([prev_page_btn, next_page_btn, preview_btn, save_region_btn]),
    widgets.HBox([x1_slider, x2_slider]),
    widgets.HBox([y1_slider, y2_slider]),
    preview_out,
    manual_out,
])

display(manual_ui)
load_page_sliders()


## Step 7: Segment Lines from Handwritten Regions

This converts each handwritten region crop into line crops. A line usually contains one medicine entry or one instruction line.


In [ ]:
# Segment handwritten region crops into line-level crops.
# Requires REGION_MANIFEST from Step 6A.
if REGION_MANIFEST.exists():
    !python3 pipeline/scripts/segment_lines.py \
      --region-manifest "{REGION_MANIFEST}" \
      --output-dir "{LINE_CROPS_DIR}" \
      --manifest-out "{LINE_MANIFEST}"
else:
    print('Missing region manifest:', REGION_MANIFEST)
    print('Run Step 6A first, or use the heuristic route output.')


## Step 8: Segment Word Crops from Lines

This creates individual word images, which is the same annotation unit used by your previous trained BD medicine-name OCR dataset.


In [ ]:
# Segment each line crop into word crops.
# Output: word images + word_manifest.csv.
if LINE_MANIFEST.exists():
    !python3 pipeline/scripts/segment_words.py \
      --line-manifest "{LINE_MANIFEST}" \
      --output-dir "{WORD_CROPS_DIR}" \
      --manifest-out "{WORD_MANIFEST}"
else:
    print('Missing line manifest:', LINE_MANIFEST)


## Step 9: Preview Word Crops

Use this preview before annotation. If many crops are poor, tune `segment_words.py` parameters or improve the line/region crops.


In [ ]:
# Display sample word crops generated by the segmentation stage.
word_images = sorted((WORD_CROPS_DIR / 'words').glob('*.png'))[:20]
print('Sample word crops:', len(word_images))
for word_img in word_images:
    print(word_img.name)
    display(IPImage(filename=str(word_img), width=180))


## Step 10: Create Word Annotation CSV

This creates `word_annotations.csv`, where each row corresponds to one cropped word image. Fill `medicine_name` only for medicine-name word crops, and mark usable rows as `reviewed`.


In [ ]:
# Create the word-level annotation sheet from word_manifest.csv.
# If you used the heuristic route, change WORD_MANIFEST to that route's word_manifest.csv first.
if WORD_MANIFEST.exists():
    !python3 pipeline/scripts/create_word_annotation_manifest.py \
      --word-manifest "{WORD_MANIFEST}" \
      --output-csv "{WORD_ANNOTATIONS}"
else:
    print('Missing word manifest:', WORD_MANIFEST)


## Step 11: Notebook Word Annotation Tool

Run this cell to open a simple in-notebook annotation UI. It shows the word crop, the source line, and the word context. Click **Save** or **Save + Next** after each annotation.

Recommended labels:

- `word_text`: exact text visible in the crop
- `medicine_name`: medicine name if this word is a medicine; otherwise keep blank
- `is_medicine`: `yes` for medicine words, `no` for non-medicine words
- `review_status`: use `reviewed` only for clean, training-ready crops


In [ ]:
# In-notebook annotation UI built with ipywidgets.
# This avoids needing Streamlit when working in Colab/Jupyter.
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage
from datetime import datetime

annotation_path = WORD_ANNOTATIONS
manifest_parent = annotation_path.parent

if not annotation_path.exists():
    raise FileNotFoundError(f'Annotation CSV not found: {annotation_path}. Run Step 10 first.')

ann_df = pd.read_csv(annotation_path).fillna('')

# Ensure expected columns exist even if the CSV was edited manually.
for col in ['word_text', 'medicine_name', 'is_medicine', 'confidence', 'annotator_id', 'review_status', 'notes', 'updated_at']:
    if col not in ann_df.columns:
        ann_df[col] = ''
ann_df['confidence'] = ann_df['confidence'].replace('', 'medium')
ann_df['review_status'] = ann_df['review_status'].replace('', 'pending')

state = {'idx': 0}

annotator_id_box = widgets.Text(value='annotator_1', description='Annotator')
word_text_box = widgets.Text(value='', description='Word')
medicine_box = widgets.Text(value='', description='Medicine')
is_medicine_box = widgets.Dropdown(options=['', 'yes', 'no'], value='', description='Is med')
confidence_box = widgets.Dropdown(options=['low', 'medium', 'high'], value='medium', description='Conf')
status_box = widgets.Dropdown(options=['pending', 'reviewed', 'reject'], value='pending', description='Status')
notes_box = widgets.Textarea(value='', description='Notes', layout=widgets.Layout(width='80%', height='70px'))
prev_button = widgets.Button(description='Previous')
next_button = widgets.Button(description='Next')
save_button = widgets.Button(description='Save', button_style='success')
save_next_button = widgets.Button(description='Save + Next', button_style='primary')
progress = widgets.Label()
out = widgets.Output()


def resolve_path(path_value):
    path = Path(str(path_value))
    return path if path.is_absolute() else (manifest_parent / path).resolve()


def load_row():
    row = ann_df.iloc[state['idx']]
    word_text_box.value = str(row.get('word_text', ''))
    medicine_box.value = str(row.get('medicine_name', ''))
    is_value = str(row.get('is_medicine', ''))
    is_medicine_box.value = is_value if is_value in ['', 'yes', 'no'] else ''
    conf_value = str(row.get('confidence', ''))
    confidence_box.value = conf_value if conf_value in ['low', 'medium', 'high'] else 'medium'
    status_value = str(row.get('review_status', ''))
    status_box.value = status_value if status_value in ['pending', 'reviewed', 'reject'] else 'pending'
    notes_box.value = str(row.get('notes', ''))
    progress.value = f"Item {state['idx'] + 1}/{len(ann_df)} | word_id={row.get('word_id', '')}"

    with out:
        clear_output(wait=True)
        print('word_id:', row.get('word_id', ''))
        print('line_id:', row.get('line_id', ''))
        for label, col, width in [
            ('Word crop', 'word_image_path', 220),
            ('Line crop', 'line_image_path', 650),
            ('Word context', 'line_context_image_path', 650),
        ]:
            path = resolve_path(row[col])
            print(label, path)
            if path.exists():
                display(IPImage(filename=str(path), width=width))
            else:
                print('Missing image:', path)


def save_row():
    idx = state['idx']
    ann_df.at[idx, 'word_text'] = word_text_box.value.strip()
    ann_df.at[idx, 'medicine_name'] = medicine_box.value.strip()
    ann_df.at[idx, 'is_medicine'] = is_medicine_box.value
    ann_df.at[idx, 'confidence'] = confidence_box.value
    ann_df.at[idx, 'review_status'] = status_box.value
    ann_df.at[idx, 'notes'] = notes_box.value.strip()
    ann_df.at[idx, 'annotator_id'] = annotator_id_box.value.strip()
    ann_df.at[idx, 'updated_at'] = datetime.utcnow().isoformat(timespec='seconds')
    ann_df.to_csv(annotation_path, index=False)


def go_prev(_):
    save_row()
    state['idx'] = max(0, state['idx'] - 1)
    load_row()


def go_next(_):
    save_row()
    state['idx'] = min(len(ann_df) - 1, state['idx'] + 1)
    load_row()


def save_only(_):
    save_row()
    progress.value = progress.value + ' | saved'


def save_and_next(_):
    save_row()
    state['idx'] = min(len(ann_df) - 1, state['idx'] + 1)
    load_row()

prev_button.on_click(go_prev)
next_button.on_click(go_next)
save_button.on_click(save_only)
save_next_button.on_click(save_and_next)

ui = widgets.VBox([
    progress,
    widgets.HBox([prev_button, next_button, save_button, save_next_button]),
    annotator_id_box,
    widgets.HBox([word_text_box, medicine_box]),
    widgets.HBox([is_medicine_box, confidence_box, status_box]),
    notes_box,
    out,
])

display(ui)
load_row()


## Step 12: Annotation Summary

Run this to check how many word crops are ready for OCR training.


In [ ]:
# Summarize annotation progress and reviewed medicine-name rows.
import pandas as pd

ann_df = pd.read_csv(WORD_ANNOTATIONS).fillna('')
print('Total word crops:', len(ann_df))
print('Annotated word_text:', (ann_df['word_text'].astype(str).str.strip() != '').sum())
print('Reviewed rows:', (ann_df['review_status'] == 'reviewed').sum())
print('Reviewed medicine rows:', ((ann_df['review_status'] == 'reviewed') & (ann_df['medicine_name'].astype(str).str.strip() != '')).sum())

display(ann_df.head(10))


## Step 13: Build Custom Word OCR Dataset

This exports reviewed medicine word crops to the same folder and CSV format as the previous BD dataset. This output can be used directly in the TrOCR training notebook.


In [ ]:
# Export reviewed medicine word crops into a TrOCR-ready dataset.
!python3 pipeline/scripts/build_ocr_dataset.py \
  --annotations-csv "{WORD_ANNOTATIONS}" \
  --output-root "{CUSTOM_WORD_DATASET}" \
  --image-path-column word_image_path \
  --label-column medicine_name \
  --approved-status reviewed \
  --seed 42

print('Custom dataset:', CUSTOM_WORD_DATASET)


## Step 13: Augment OCR Training Split

This creates `data/custom_word_ocr_dataset_augmented`. Only the training split is augmented; validation and testing are copied unchanged.


In [ ]:
# Train-only OCR augmentation for TrOCR.
!python3 pipeline/scripts/augment_ocr_dataset.py \
  --input-root "{CUSTOM_WORD_DATASET}" \
  --output-root "{AUGMENTED_WORD_DATASET}" \
  --augmentations-per-image 3 \
  --seed 42

print('Augmented OCR dataset:', AUGMENTED_WORD_DATASET)
print('Use this in the TrOCR notebook as MANUAL_DATASET_BASE.')


## Step 14: Preview Exported Dataset

Confirm that the generated dataset has the expected split CSV files and word images.


In [ ]:
# Inspect the exported OCR dataset structure.
for dataset_root in [CUSTOM_WORD_DATASET, AUGMENTED_WORD_DATASET]:
    print('
Dataset:', dataset_root)
    for split, csv_name, img_dir in [
        ('Training', 'training_labels.csv', 'training_words'),
        ('Validation', 'validation_labels.csv', 'validation_words'),
        ('Testing', 'testing_labels.csv', 'testing_words'),
    ]:
        csv_path = dataset_root / split / csv_name
        image_path = dataset_root / split / img_dir
        print(split, 'CSV exists:', csv_path.exists(), 'images:', len(list(image_path.glob('*.png'))) if image_path.exists() else 0)
        if csv_path.exists():
            display(pd.read_csv(csv_path).head())


## Next Notebook

After this notebook creates the augmented dataset, open `phase4_trocr_word_level.ipynb`. The OCR notebook now auto-prefers:

```python
data/custom_word_ocr_dataset_augmented
```

You can also set it manually before dataset discovery:

```python
MANUAL_DATASET_BASE = 'data/custom_word_ocr_dataset_augmented'
```
